# Create PCF Awards (Prostate Cancer Foundation research awards)

**Source:** PCF's "The Work We Fund" pages, one page per funded project under
https://www.pcf.org/our-impact/the-work-we-fund/<program>/<class>/<slug>/, enumerated
from the WordPress page sitemap (https://www.pcf.org/page-sitemap.xml). Programs:
Young Investigator Awards (classes 2020-2025), Challenge Awards (2019-2025), TACTICAL
Awards (2022, 2025), Creativity Awards (2009-2011, 2016) and PCF-VA (VAlor) awards.
Each page gives the project title, investigators (Principal Investigators /
Co-Investigators with institutions in parentheses, or the Young Investigator with
institution and mentors) and a description. Named-award headings ("2024 Neil DeFeo -
PCF Young Investigator Award") are kept in the raw table as `named_award`.

**Scope:** all published programs are research awards. **Excluded:** the 2012 SU2C-PCF
Prostate Dream Teams (co-funded with Stand Up To Cancer / AACR and published only as
press-release prose, with no structured title or PI). PCF says it has funded 2,250+
projects since 1993. Its site only lists these classes, so coverage is partial
(mostly 2019-2025).

**314 awards, class years 2009-2025** (YI 181, Challenge 91, Creativity 36, TACTICAL 6;
100% title / PI / description, 98% PI institution; local run 2026-09-30). PCF-VA
listing pages have no per-award detail pages, so there are none from that program.

**`funder_award_id` (section 2.1.1):** citing works use PCF's internal award numbers
("17CHAL06", "18YOUN19", "20CHAS01", "22TACT01"; ~442 existing F4320306163 award
stubs, mixed with NIH/DoD numbers). PCF does not print these on the award pages and
they cannot be derived (the sequence number is not published), so the key is
`PCF-<WordPress page id>` (page shortlink ?p=<id>). It collapses onto 0 existing stubs.

**No amounts** are published per award (PCF gives program-level ranges only), so
amount/currency are NULL (Step 6.7 waived). Dates: only the award class year is
published and is used as `start_year`, with `start_date` = Jan 1 of that year.
Mentors of Young Investigators are kept in the raw `mentors` JSON but not emitted
as investigators.

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320306163`
- display_name: Prostate Cancer Foundation
- ror_id / doi: from `openalex.funders.funders` (ror 01npwtv09, doi 10.13039/100000892)
- Not to be confused with Prostate Cancer Foundation of Australia (F4320320370).

**Prerequisites:** run `scripts/local/pcf_to_s3.py` first. It uploads
`s3://openalex-ingest/awards/pcf/pcf_projects.parquet` (section 1.4 shrink guard).

**Priority:** `516` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.pcf_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/pcf/pcf_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_grants, COUNT(DISTINCT funder_award_id) as distinct_award_ids FROM openalex.awards.pcf_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.pcf_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.pcf_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320306163 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320306163;

## Step 2: Create PCF Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.pcf_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320306163  -- Prostate Cancer Foundation
),

g AS (
    SELECT
        r.*,
        -- struct field order MUST match openalex_awards_raw: ...orcid, role_start, affiliation.
        CASE WHEN NULLIF(TRIM(r.pi_family_name), '') IS NOT NULL THEN named_struct(
            'given_name', NULLIF(TRIM(r.pi_given_name), ''),
            'family_name', NULLIF(TRIM(r.pi_family_name), ''),
            'orcid', CAST(NULL AS STRING),
            'role_start', CAST(NULL AS DATE),
            'affiliation', CASE WHEN NULLIF(TRIM(r.pi_institution), '') IS NOT NULL THEN named_struct(
                'name', NULLIF(TRIM(r.pi_institution), ''),
                'country', CAST(NULL AS STRING),
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            ) END
        ) END AS lead,
        filter(
            transform(
                COALESCE(from_json(r.co_investigators,
                    'ARRAY<STRUCT<name:STRING, given_name:STRING, family_name:STRING, institution:STRING>>'),
                    array()),
                p -> CASE WHEN NULLIF(TRIM(p.family_name), '') IS NOT NULL THEN named_struct(
                    'given_name', NULLIF(TRIM(p.given_name), ''),
                    'family_name', NULLIF(TRIM(p.family_name), ''),
                    'orcid', CAST(NULL AS STRING),
                    'role_start', CAST(NULL AS DATE),
                    'affiliation', CASE WHEN NULLIF(TRIM(p.institution), '') IS NOT NULL THEN named_struct(
                        'name', NULLIF(TRIM(p.institution), ''),
                        'country', CAST(NULL AS STRING),
                        'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
                    ) END
                ) END
            ),
            x -> x IS NOT NULL
        ) AS co
    FROM openalex.awards.pcf_raw r
    WHERE r.funder_award_id IS NOT NULL AND TRIM(r.funder_award_id) != ''
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    NULLIF(TRIM(g.description), '') as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    CAST(NULL AS DOUBLE) as amount,      -- PCF publishes no per-award amounts
    CAST(NULL AS STRING) as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    'research' as funding_type,
    CONCAT('PCF ', g.program) as funder_scheme,
    'pcf_work_we_fund' as provenance,
    TRY_TO_DATE(CONCAT(g.class_year, '-01-01'), 'yyyy-MM-dd') as start_date,
    CAST(NULL AS DATE) as end_date,
    TRY_CAST(g.class_year AS INT) as start_year,
    CAST(NULL AS INT) as end_year,
    g.lead as lead_investigator,
    try_element_at(g.co, 1) as co_lead_investigator,
    CASE WHEN g.lead IS NOT NULL THEN concat(array(g.lead), g.co)
         WHEN size(g.co) > 0 THEN g.co END as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM g
CROSS JOIN src_funder f;

In [ ]:
%sql
-- Shape check (section 2.1.1): every funder_award_id is PCF-<wordpress page id>.
SELECT assert_true(
    COUNT_IF(NOT funder_award_id RLIKE '^PCF-[0-9]+$') = 0
    AND COUNT(*) = COUNT(DISTINCT id),
    'pcf_awards: unexpected funder_award_id shape or duplicate ids'
) AS shape_ok
FROM openalex.awards.pcf_awards;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'pcf_work_we_fund' AND priority = 516;

-- Insert into openalex_awards_raw with priority.
-- Priority 516: direct-from-funder ingest of PCF's "The Work We Fund" award pages.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    516 as priority
FROM openalex.awards.pcf_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.pcf_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, start_year,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name, size(investigators) as n_inv
FROM openalex.awards.pcf_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, start_year, COUNT(*) as cnt
FROM openalex.awards.pcf_awards
GROUP BY funder_scheme, start_year ORDER BY funder_scheme, start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: PI names and titles.
SELECT lead_investigator.given_name, lead_investigator.family_name, COUNT(*) AS n
FROM openalex.awards.pcf_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.pcf_awards
GROUP BY 1 ORDER BY n DESC LIMIT 10;

In [ ]:
%sql
-- Section 6.7 coverage.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(start_year) as has_start_year,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.pcf_awards;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'pcf_work_we_fund'
GROUP BY provenance, priority;